# Training step 1: check of the curved-sky J3 loss

`src/sht_torch.py` (differentiable spin-2 transforms on the CAR cut-out) and `src/losses_car.py` (curved-sky J3), as planned in `docs/04_training_plan.md`. Checks:

1. gradients (`torch.autograd.gradcheck`) of the transforms and of the whole loss;
2. adjoints on the dataset geometry;
3. normalization of both loss terms on real validation maps (cluster dataset);
4. band limit: what part of the network output the loss sees;
5. computing time of one training step.

Data: the 100 validation maps copied from the cluster (`results/task2/cluster/`).

In [1]:
import os, sys, time
import numpy as np
import healpy as hp
import torch
from pixell import enmap, curvedsky, utils

REPO = os.path.abspath('..')
sys.path.insert(0, os.path.join(REPO, 'src'))
from config_loader import load_config
cfg = load_config(os.path.join(REPO, 'src', 'config.dict'), create_dirs=False)   # cluster config: only scalars are used here
sys.path.append(os.path.join(REPO, '..', 'CMBpipeline', 'source'))                 # local CMBpipeline copy (the config points to the cluster one)
import make_dataset as dd
from sht_torch import CarSHT
from losses_car import CarLossJ3
import network_2d                                                                  # ../CMBpipeline/source (read-only)

CL  = os.path.join(REPO, 'results', 'task2', 'cluster')
tag = cfg['dataset_tag']
mask_car = enmap.read_map(f'{CL}/aux/mask_car_{tag}.fits')
var_car  = enmap.read_map(f'{CL}/aux/variance_car_{tag}.fits')
cltt, clee, clbb = np.load(f'{CL}/aux/cls_signal_r0.034.npy')
seeds = np.load(f'{CL}/aux/seeds_{tag}.npz')
X_valid = torch.load(f'{CL}/data/valid_{tag}.pt')[:5]
factor, L, fwhm = cfg['map_rescale_factor'], cfg['sht_lmax'], cfg['fwhm_rad']
NT = 8
shape, wcs = mask_car.shape, mask_car.wcs
ny, nx = shape; NPIX = ny * nx
inside = np.asarray(mask_car) > 0; NOBS = int(inside.sum())
print(f'factor {factor}, sht lmax {L}, method {cfg["sht_method"]}, cut-out {nx} x {ny}, observed pixels {NOBS}, threads {NT}')

factor 0.75068694, sht lmax 1124, method cyl, cut-out 1120 x 320, observed pixels 286159, threads 8


## 1. Gradients

`gradcheck` in float64 on a small CAR cut-out (2° fejer1 grid, rows 20–35 and columns 30–60, ℓ ≤ 40), for both pixell methods, then for the whole loss.

In [2]:
fshape, fwcs = enmap.fullsky_geometry(res=2 * utils.degree, variant='fejer1')
sshape, swcs = enmap.slice_geometry(fshape, fwcs, (slice(20, 36), slice(30, 60)))
torch.manual_seed(0)
for meth in ('cyl', '2d'):
    s = CarSHT(sshape, swcs, lmax=40, method=meth)
    f = torch.randn((2,) + tuple(sshape), dtype=torch.float64, requires_grad=True)
    a = torch.randn((2, s.nalm), dtype=torch.complex128)
    a[:, s.m == 0] = a[:, s.m == 0].real.to(torch.complex128)        # m = 0 coefficients are real
    a.requires_grad_(True)
    ok_A = torch.autograd.gradcheck(lambda x: s.analysis(x), (f,), eps=1e-6, atol=1e-8)
    # synthesis ignores Im a_l0, so check it on the real and imaginary parts of the m > 0 coefficients
    re, im = a.real.detach().clone().requires_grad_(True), a.imag.detach().clone()
    im[:, s.m == 0] = 0; im.requires_grad_(True)
    def syn(re, im):
        im0 = torch.where(torch.tensor(s.m == 0), torch.zeros_like(im), im)
        return s.synthesis(torch.complex(re, im0))
    ok_Y = torch.autograd.gradcheck(syn, (re, im), eps=1e-6, atol=1e-8)
    print(f'{meth:3s}: gradcheck analysis {ok_A}, synthesis {ok_Y}')

# whole loss
s = CarSHT(sshape, swcs, lmax=40, method='cyl')
bl_s = hp.gauss_beam(np.radians(5), lmax=40)
lossf = CarLossJ3(s, bl_s, clee, clbb, factor, 'cpu', dtype=torch.float64)
y_true = torch.randn((1, 2) + tuple(sshape), dtype=torch.float64)
msk = (torch.rand(tuple(sshape)) > 0.3).double()[None]
inh = (0.5 + torch.rand(tuple(sshape))).double()[None] * msk
yp = (1e2 * torch.randn((1, 2) + tuple(sshape), dtype=torch.float64)).requires_grad_(True)
# J3 is quadratic in y, so central differences are exact for any step; eps = 1e-2 avoids round-off on inputs ~ 1e2
print('gradcheck of the whole J3:', torch.autograd.gradcheck(lambda y: lossf(y_true, y, msk, inh), (yp,), eps=1e-2, atol=1e-8, rtol=1e-6))

cyl: gradcheck analysis True, synthesis True


2d : gradcheck analysis True, synthesis True


gradcheck of the whole J3: True


## 2. Adjoints on the dataset geometry

Dot tests ⟨A f, a⟩ = ⟨f, Aᵀ a⟩ and ⟨Y a, f⟩ = ⟨a, Yᵀ f⟩ on the 1120 × 320 cut-out (ℓ ≤ 1124, `'cyl'`), with the inner product ⟨a, b⟩ = Σ w_m Re(a* b).

In [3]:
sht = CarSHT(shape, wcs, lmax=L, method=cfg['sht_method'], nthread=NT)
rng = np.random.default_rng(0)
f = rng.standard_normal((2,) + shape)
a = rng.standard_normal((2, sht.nalm)) + 1j * rng.standard_normal((2, sht.nalm)); a[:, sht.m == 0] = a[:, sht.m == 0].real
adot = lambda x, y: np.sum(sht.w * (np.conj(x) * y).real)
for name, lhs, rhs in (('A', adot(sht.A(f), a), np.sum(f * sht.AT(a))), ('Y', np.sum(sht.Y(a) * f), adot(a, sht.YT(f)))):
    print(f'{name}: {lhs:.12e} vs {rhs:.12e}, relative difference {abs(lhs - rhs) / abs(lhs):.1e}')

A: 6.288486073009e-02 vs 6.288486073009e-02, relative difference 3.7e-14
Y: 2.062633867992e+05 vs 2.062633867992e+05, relative difference 1.4e-16


## 3. Normalization of the two terms on real data

Validation map j has CMB seed 1000 + j and noise seed 101000 + j, so its signal can be rebuilt exactly. First check that `make_dataset` reproduces the stored map, then evaluate the loss at ŷ = the true (unbeamed) signal, normalized as in the loader. Expected values:
- **likelihood:** E[Σ m n²/σ²] = N_obs for Q and for U, so term 1 ≈ 2 N_obs / N_pix;
- **prior:** for a field drawn from C_ℓ and band-limited to ℓ ≤ L on a cut-out of sky fraction f_sky, Σ w |a|²/C ≈ f_sky · Σ_{ℓ=2}^{L} (2ℓ+1) for E and for B.

In [4]:
gen  = dd.make_dataset(0, 0, 0, True, False)
ps   = gen._create_spectrum_array(cltt, clee, clbb)
gen0 = dd.make_dataset(0, 0, 0, False, False)                # no beam: the Wiener-filter target
loss_car = CarLossJ3(sht, hp.gauss_beam(fwhm, lmax=L), clee, clbb, factor, 'cpu')
fsky = float(np.sum(mask_car.pixsizemap()) / (4 * np.pi))
ell = np.arange(L + 1)
exp1 = 2 * NOBS / NPIX
exp2_bl = 2 * fsky * np.sum(2 * ell[2:] + 1) / NPIX
print(f'f_sky(cut-out) = {fsky:.4f}; expected term1 = {exp1:.4f}; expected term2 (band-limited signal) = {exp2_bl:.4f}')

ps_bl = ps.copy(); ps_bl[..., L + 1:] = 0                     # same realization, cut at l <= L
def norm(x, ref):                                              # loader normalization (mean of the observed map)
    return (x - ref.mean(axis=(-2, -1), keepdims=True)) * factor
for j in range(3):
    cs, ns = int(seeds['cmb_seeds_valid'][j]), int(seeds['noise_seeds_valid'][j])
    dQ, dU = gen.make_inho_maps_car(ps, var_car, mask_car, shape, wcs, cs, ns)
    same = np.allclose(np.stack([dQ, dU]).astype(np.float32), X_valid[j, :2].numpy(), atol=1e-6, rtol=0)
    obs = X_valid[j, :2].numpy().astype(np.float64)
    d = torch.tensor(norm(obs, obs)[None], dtype=torch.float32)
    sQ, sU = gen0.get_QUmaps_car(gen0.get_alm(ps, cs), shape, wcs)              # true unbeamed signal (l <= 1535)
    sig = np.stack([sQ, sU])
    bQ, bU = gen0.get_QUmaps_car(curvedsky.rand_alm(ps_bl, lmax=gen0.lmax_sim, seed=cs), shape, wcs)
    sig_bl = np.stack([bQ, bU])                                 # same realization band-limited to l <= L (prior check)
    m_t, v_t = X_valid[j, 2:3], X_valid[j, 3:4]
    with torch.no_grad():
        for name, y in (('y = true signal (l <= 1535)', sig), ('y = signal band-limited to l <= 1124', sig_bl), ('y = 0', 0 * sig)):
            yt = torch.tensor((y - obs.mean(axis=(-2, -1), keepdims=True))[None] * factor, dtype=torch.float32)
            t1, t2 = loss_car.terms(d, yt, m_t, v_t)
            print(f'map {j} (reproduced: {same}) {name:38s}: term1 = {t1.item():.4f} ({t1.item()/exp1:.4f} x exp.), term2 = {t2.item():.4f}'
                  + (f' ({t2.item()/exp2_bl:.4f} x exp.)' if 'band' in name else '') + f', J3 = {(t1 + t2).item():.4f}')
print(f'mean of the observed Q map (removed by the loader): {X_valid[0, 0].mean():.2e} muK, rms noise in mask {np.sqrt(var_car[inside].mean()):.3f} muK')

f_sky(cut-out) = 0.1414; expected term1 = 1.5969; expected term2 (band-limited signal) = 0.9990


map 0 (reproduced: True) y = true signal (l <= 1535)           : term1 = 1.6070 (1.0063 x exp.), term2 = 7.3198, J3 = 8.9267


map 0 (reproduced: True) y = signal band-limited to l <= 1124  : term1 = 1.5946 (0.9986 x exp.), term2 = 1.4578 (1.4593 x exp.), J3 = 3.0524
map 0 (reproduced: True) y = 0                                 : term1 = 39.5118 (24.7433 x exp.), term2 = 0.0003, J3 = 39.5121


map 1 (reproduced: True) y = true signal (l <= 1535)           : term1 = 1.6055 (1.0054 x exp.), term2 = 7.2625, J3 = 8.8680
map 1 (reproduced: True) y = signal band-limited to l <= 1124  : term1 = 1.5933 (0.9978 x exp.), term2 = 1.4159 (1.4174 x exp.), J3 = 3.0092


map 1 (reproduced: True) y = 0                                 : term1 = 39.2279 (24.5655 x exp.), term2 = 0.0010, J3 = 39.2289


map 2 (reproduced: True) y = true signal (l <= 1535)           : term1 = 1.6048 (1.0050 x exp.), term2 = 7.3014, J3 = 8.9062
map 2 (reproduced: True) y = signal band-limited to l <= 1124  : term1 = 1.5924 (0.9972 x exp.), term2 = 1.4308 (1.4323 x exp.), J3 = 3.0232


map 2 (reproduced: True) y = 0                                 : term1 = 39.7716 (24.9060 x exp.), term2 = 0.0004, J3 = 39.7720
mean of the observed Q map (removed by the loader): 1.01e-01 muK, rms noise in mask 0.338 muK


**Reading.** The term-1 excess at the true signal measures how well the loss's beam operator Y B_ℓ A reproduces the data's beamed signal. Signal power between ℓ = 1125 and 1535, which the 9.6′ grid cannot resolve exactly, aliases into ℓ ≤ 1124 (about 1.8 % rms of the beamed signal, measured while preparing the plan). A Wiener-filter output has almost no power at those multipoles, so this matters only for this test.

## 4. Band limit: which part of the network output the loss sees

The loss depends on ŷ only through a = A ŷ (ℓ ≤ 1124), so the Wiener-filtered map is defined as Y A ŷ. Checks:
- **(a)** Y A ŷ against ŷ, inside the mask;
- **(b)** whether Y A acts as a projector on the cut-out: Y A (Y A ŷ) against Y A ŷ.

Both are evaluated for the true signal and for the output of an untrained network with CMBpipeline's selected architecture.

In [5]:
filters = [16, 40, 64, 152, 152, 152]          # CMBpipeline best trial (study ..._res_arcmin, trial 3)
torch.manual_seed(0)
model = network_2d.DeepWiener_threechannels(2, 2, filters)
xin = torch.cat((torch.tensor(norm(obs, obs)[None], dtype=torch.float32), X_valid[j:j+1, 2:4]), dim=1)
with torch.no_grad():
    y_net = model(xin)[0].numpy().astype(np.float64)
rel = lambda x, y: np.sqrt(np.mean((x - y)[:, inside]**2) / np.mean(y[:, inside]**2))
for name, y in (('true signal', sig * factor), ('untrained network output', y_net)):
    p1 = sht.Y(sht.A(y)); p2 = sht.Y(sht.A(p1))
    print(f'{name:26s}: |YA y - y| / |y| = {rel(p1, y):.3e};  |YA(YA y) - YA y| / |YA y| = {rel(p2, p1):.3e}')

true signal               : |YA y - y| / |y| = 3.613e-01;  |YA(YA y) - YA y| / |YA y| = 9.633e-03


untrained network output  : |YA y - y| / |y| = 2.638e-01;  |YA(YA y) - YA y| / |YA y| = 6.259e-03


## 5. Computing time of one training step (CPU, this machine)

CMBpipeline's best architecture, batch 1. Forward pass of the network, the loss (A, Y), then backward (Aᵀ, Yᵀ and the network). Each part is timed separately.

In [6]:
opt = torch.optim.Adam(model.parameters(), lr=5e-5)
yt = torch.tensor(norm(obs, obs)[None], dtype=torch.float32)
m_t, v_t = X_valid[0, 2:3], X_valid[0, 3:4]
for it in range(4):
    opt.zero_grad(); sht.reset_timer()
    t0 = time.perf_counter(); pred = model(xin); t1 = time.perf_counter()
    l = loss_car(yt, pred, m_t, v_t); t2 = time.perf_counter(); t_fwd_sht = sht.t_total
    l.backward(); t3 = time.perf_counter(); opt.step()
    if it > 0:
        print(f'step {it}: network forward {t1-t0:.2f} s | loss forward {t2-t1:.2f} s (A + Y: {t_fwd_sht:.2f} s) | '
              f'backward {t3-t2:.2f} s (A^T + Y^T: {sht.t_total - t_fwd_sht:.2f} s) | total {t3-t0:.2f} s, transforms {sht.t_total:.2f} s ({sht.n_calls} calls)')

step 1: network forward 1.04 s | loss forward 0.14 s (A + Y: 0.13 s) | backward 2.97 s (A^T + Y^T: 0.13 s) | total 4.16 s, transforms 0.26 s (4 calls)


step 2: network forward 0.95 s | loss forward 0.15 s (A + Y: 0.13 s) | backward 2.83 s (A^T + Y^T: 0.13 s) | total 3.93 s, transforms 0.26 s (4 calls)


step 3: network forward 0.95 s | loss forward 0.15 s (A + Y: 0.14 s) | backward 2.84 s (A^T + Y^T: 0.13 s) | total 3.94 s, transforms 0.26 s (4 calls)
